# 01 Ingest - Azure Data Factory Simulation

This notebook simulates an Azure Data Factory ingestion pattern for NYC yellow taxi parquet data. It downloads a parquet file from a public endpoint and uploads it into Azure Data Lake Storage Gen2 using the Azure Blob SDK.

Design notes:
- Uses environment variables for all runtime configuration.
- Adds request headers and retry logic for resilient ingestion.
- Writes to a realistic raw landing path in Azure storage.

## Imports and configuration

In [ ]:
import os
import time
from pathlib import Path

import requests
from azure.storage.blob import BlobServiceClient

from src.config import AZURE_CONFIG

SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
FILE_NAME = "yellow_tripdata_2023-01.parquet"
LOCAL_DOWNLOAD_DIR = Path(os.getenv("LOCAL_DOWNLOAD_DIR", "./data/raw"))
LOCAL_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_FILE_PATH = LOCAL_DOWNLOAD_DIR / FILE_NAME

HEADERS = {
    "User-Agent": "nyc-taxi-pipeline/1.0",
    "Accept": "application/octet-stream",
}

MAX_RETRIES = int(os.getenv("INGEST_MAX_RETRIES", "3"))
BACKOFF_SECONDS = int(os.getenv("INGEST_BACKOFF_SECONDS", "5"))
TARGET_BLOB_PATH = f"{AZURE_CONFIG.raw_prefix.strip('/')}/{FILE_NAME}"

print(f"Raw landing path: {TARGET_BLOB_PATH}")

## Download source parquet with retry logic

In [ ]:
def download_with_retry(url: str, destination: Path, headers: dict, max_retries: int = 3, backoff_seconds: int = 5) -> Path:
    """Download a file with simple exponential backoff retry logic."""
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            with requests.get(url, headers=headers, stream=True, timeout=60) as response:
                response.raise_for_status()
                with open(destination, "wb") as file_handle:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            file_handle.write(chunk)
            return destination
        except requests.RequestException as exc:
            last_error = exc
            print(f"Attempt {attempt} failed: {exc}")
            if attempt < max_retries:
                sleep_for = backoff_seconds * attempt
                print(f"Retrying in {sleep_for} seconds...")
                time.sleep(sleep_for)

    raise RuntimeError(f"Download failed after {max_retries} attempts") from last_error


downloaded_file = download_with_retry(
    url=SOURCE_URL,
    destination=LOCAL_FILE_PATH,
    headers=HEADERS,
    max_retries=MAX_RETRIES,
    backoff_seconds=BACKOFF_SECONDS,
)

print(f"Downloaded file to: {downloaded_file}")

## Upload file to Azure Data Lake Storage Gen2

In [ ]:
if not AZURE_CONFIG.blob_connection_string:
    raise ValueError("AZURE_BLOB_CONNECTION_STRING is not set. Configure it before upload.")

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONFIG.blob_connection_string)
blob_client = blob_service_client.get_blob_client(
    container=AZURE_CONFIG.container,
    blob=TARGET_BLOB_PATH,
)

with open(downloaded_file, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)

mock_abfss_path = (
    f"abfss://{AZURE_CONFIG.container}@{AZURE_CONFIG.storage_account}"
    f".dfs.core.windows.net/{TARGET_BLOB_PATH}"
)

print(f"Uploaded raw dataset to: {mock_abfss_path}")